In [9]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
import os
import getpass

if "GOOGLE_API_KEY" not in os.environ:
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter your Google AI API key: ")

In [10]:
model = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=1.0,  # Gemini 3.0+ defaults to 1.0
    max_tokens=None,
    timeout=None,
    max_retries=2,
    # other params...
)

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


In [11]:
messages = [
    (
        "system",
        "You are a helpful assistant that translates English to English. Translate the user sentence.",
    ),
    ("human", "I love programming."),
]
ai_msg = model.invoke(messages)
ai_msg

AIMessage(content='I am passionate about programming.', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019c9e64-0451-79c1-b010-3d6429ec7ce8-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 21, 'output_tokens': 258, 'total_tokens': 279, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 252}})

In [18]:
class BlogState(TypedDict):
    title: str
    outline: str
    content: str
    evaluate: str

In [19]:
def create_outline(state: BlogState) -> BlogState:

    # fetch title
    title = state['title']

    # call llm gen outline
    prompt = f'Generate a detailed outline for a blog on the topic - {title}'
    outline = model.invoke(prompt).content

    # update state
    state['outline'] = outline

    return state

In [20]:
def create_blog(state: BlogState) -> BlogState:

    title = state['title']
    outline = state['outline']

    prompt = f'Write a detailed blog on the title - {title} using the following outline \n {outline}'

    content = model.invoke(prompt).content

    state['content'] = content

    return state

In [21]:
def evaluate_sample(state: BlogState) -> BlogState:

    outline = state['outline']
    content = state['content']

    prompt = f'Based on this outline \n {outline} rate my blog - {content} out of 100'

    evaluate = model.invoke(prompt).content

    state['evaluate'] = evaluate

    return state

In [22]:
graph = StateGraph(BlogState)

# nodes
graph.add_node('create_outline', create_outline)
graph.add_node('create_blog', create_blog)
graph.add_node('evaluate_sample', evaluate_sample)

# edges
graph.add_edge(START, 'create_outline')
graph.add_edge('create_outline', 'create_blog')
graph.add_edge('create_blog', 'evaluate_sample')
graph.add_edge('evaluate_sample', END)

# compile 
workflow = graph.compile()

In [23]:
initial_state = {'title': 'Viart Kohli'}

final_state = workflow.invoke(initial_state)

print(final_state['evaluate'])

This blog post is absolutely **outstanding!**

**Rating: 98/100**

Here's a breakdown of why it's so good:

**Strengths:**

1.  **Impeccable Outline Adherence:** You have followed the detailed outline almost to the letter. Every major section and almost every sub-point has been addressed comprehensively and logically. This is a huge win for structuring a complex topic.
2.  **Engaging and Evocative Language:** The writing is far from dry. Phrases like "symphony of elegance and aggression," "cultural touchstone," "monument of records," and "relentless pursuit of excellence" keep the reader hooked. You've struck a perfect balance between factual information and compelling storytelling.
3.  **Comprehensive Coverage:** You've left no stone unturned. From his early life and U-19 triumph to his batting records, captaincy, struggles, comeback, global icon status, and legacy, every facet of Kohli's career and impact is covered in depth.
4.  **Strong Narrative Flow:** The blog post flows seamles